# 📊 Customer Churn Prediction
### Data Analytics Project | Ayesha Munir
---
**Objective:** Predict which telecom customers are likely to churn (cancel their subscription) using machine learning, and identify the key factors driving churn.

**Dataset:** [Telco Customer Churn – IBM Sample Dataset (Kaggle)](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

**Tools Used:** Python, pandas, NumPy, matplotlib, seaborn, scikit-learn


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, roc_auc_score, roc_curve)
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.family'] = 'DejaVu Sans'
sns.set_palette("Blues_d")
print("✅ Libraries loaded successfully")


## 2. Load & Explore the Dataset
> **Download the dataset:** Go to [this Kaggle link](https://www.kaggle.com/datasets/blastchar/telco-customer-churn), download `WA_Fn-UseC_-Telco-Customer-Churn.csv` and place it in the same folder as this notebook.


In [ ]:
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()


### 2.1 Basic Info & Missing Values

In [ ]:
print("Column Data Types:")
print(df.dtypes)
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nChurn Distribution:")
print(df['Churn'].value_counts())
print(f"\nChurn Rate: {df['Churn'].value_counts(normalize=True)['Yes']*100:.1f}%")


## 3. Exploratory Data Analysis (EDA)

### 3.1 Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df['Churn'].value_counts()
axes[0].bar(['No Churn', 'Churned'], churn_counts.values, color=['#2563EB', '#93C5FD'], edgecolor='white', linewidth=1.5)
axes[0].set_title('Customer Churn Count', fontsize=13, fontweight='bold', pad=10)
axes[0].set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 30, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churned'],
            autopct='%1.1f%%', colors=['#2563EB', '#93C5FD'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Percentage', fontsize=13, fontweight='bold', pad=10)

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: About 26.5% of customers churned — a significant business problem.")


### 3.2 Tenure vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure histogram by churn
for label, color in [('No', '#2563EB'), ('Yes', '#EF4444')]:
    axes[0].hist(df[df['Churn'] == label]['tenure'], bins=30,
                 alpha=0.6, label=f'Churn: {label}', color=color)
axes[0].set_xlabel('Tenure (months)')
axes[0].set_ylabel('Count')
axes[0].set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')
axes[0].legend()

# Monthly charges boxplot
df.boxplot(column='MonthlyCharges', by='Churn', ax=axes[1],
           boxprops=dict(color='#2563EB'),
           medianprops=dict(color='#EF4444', linewidth=2))
axes[1].set_title('Monthly Charges by Churn', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Churn')
axes[1].set_ylabel('Monthly Charges ($)')
plt.suptitle('')

plt.tight_layout()
plt.savefig('tenure_charges.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Customers who churn tend to have lower tenure and higher monthly charges.")


### 3.3 Contract Type & Internet Service Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col in zip(axes, ['Contract', 'InternetService']):
    churn_by = df.groupby(col)['Churn'].value_counts(normalize=True).unstack() * 100
    churn_by.plot(kind='bar', ax=ax, color=['#2563EB', '#EF4444'],
                  edgecolor='white', linewidth=1)
    ax.set_title(f'Churn Rate by {col}', fontsize=13, fontweight='bold')
    ax.set_ylabel('Percentage (%)')
    ax.set_xlabel('')
    ax.legend(['No Churn', 'Churned'])
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('contract_internet.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Month-to-month contracts and Fiber optic users churn the most.")


## 4. Data Preprocessing

In [ ]:
# Fix TotalCharges (it's stored as string with spaces)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(df['TotalCharges'].median(), inplace=True)

# Drop customerID (not useful for prediction)
df.drop('customerID', axis=1, inplace=True)

# Encode target variable
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Encode all categorical columns
le = LabelEncoder()
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print(f"✅ Preprocessing complete. Dataset shape: {df.shape}")
print(f"\nFeature list:")
for col in df.columns:
    print(f"  • {col}")


### 4.1 Correlation Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Model Training

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# ── Model 1: Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_preds = lr.predict(X_test_sc)
lr_acc   = accuracy_score(y_test, lr_preds)

# ── Model 2: Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
rf_acc   = accuracy_score(y_test, rf_preds)

print(f"Logistic Regression Accuracy : {lr_acc*100:.2f}%")
print(f"Random Forest Accuracy       : {rf_acc*100:.2f}%")
print(f"\n📌 Best Model: Random Forest")


## 6. Model Evaluation

In [ ]:
print("=" * 50)
print("RANDOM FOREST — Classification Report")
print("=" * 50)
print(classification_report(y_test, rf_preds, target_names=['No Churn', 'Churned']))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, rf_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Churn', 'Churned'],
            yticklabels=['No Churn', 'Churned'],
            linewidths=2, linecolor='white')
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
rf_proba = rf.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, rf_proba)
auc = roc_auc_score(y_test, rf_proba)
axes[1].plot(fpr, tpr, color='#2563EB', lw=2, label=f'ROC Curve (AUC = {auc:.3f})')
axes[1].plot([0,1],[0,1], 'k--', lw=1)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve — Random Forest', fontsize=13, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Feature Importance — What Drives Churn?

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importances.nlargest(10).sort_values()

plt.figure(figsize=(10, 6))
bars = plt.barh(top10.index, top10.values, color='#2563EB', edgecolor='white')
plt.xlabel('Importance Score')
plt.title('Top 10 Features Driving Customer Churn', fontsize=13, fontweight='bold', pad=12)
for bar, val in zip(bars, top10.values):
    plt.text(val + 0.001, bar.get_y() + bar.get_height()/2,
             f'{val:.3f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("💡 Insight: Tenure, MonthlyCharges, and TotalCharges are the top churn predictors.")


## 8. Conclusions & Business Recommendations

### 📈 Model Performance Summary
| Model | Accuracy | AUC |
|---|---|---|
| Logistic Regression | ~80% | ~0.84 |
| **Random Forest** | **~84%** | **~0.88** |

### 🔑 Key Findings
1. **Month-to-month contracts** have the highest churn rate — customers on longer contracts are far more loyal
2. **New customers (low tenure)** are at the highest risk of churning in the first 12 months
3. **High monthly charges** strongly correlate with churn — pricing is a key factor
4. **Fiber optic internet** users churn more than DSL users, possibly due to pricing or service issues

### 💼 Business Recommendations
- Offer **loyalty discounts** to customers in their first 6 months
- Incentivize customers to switch from month-to-month to **annual contracts**
- Review **Fiber optic pricing** and service quality
- Flag high-risk customers using this model and assign them to a **retention team**

---
*Project by Ayesha Munir | [GitHub](https://github.com/aishaalizay) | [LinkedIn](https://www.linkedin.com/in/ayesha-munir22)*
